# DealScout - Phase 1, Step 2: Data Pre-processing

Generate concise product summaries for every `Item` using an LLM.

In [ ]:
import os
from dotenv import load_dotenv

from dealscout.core.items import Item
from dealscout.models.batch import Batch

load_dotenv(override=True)

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

In [ ]:
LITE_MODE = True

In [ ]:
username = "Ankit-Singh-ai"
dataset = f"{username}/dealscout_items_raw_lite" if LITE_MODE else f"{username}/dealscout_items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

In [ ]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [ ]:
Batch.create(items, LITE_MODE)
Batch.run()
Batch.fetch()

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets 

In [ ]:
username = "Ankit-Singh-ai"
full = f"{username}/dealscout_items_full"
lite = f"{username}/dealscout_items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)